# Infrastructure diagnostics — offline

Owner: repository founder. Updated 2026-09-20.

This notebook is the engineer companion to the [QR research workflow](research-foundations.ipynb). It answers whether the installed runtime, configured reference inputs, and existing acquisition, publication, capture, replay, and supervision consumers still meet their bounded contracts.

The default run is completely offline. It uses synthetic wire and archive responses, copies only the fixtures required by the retained consumers into an independent TemporaryDirectory, and never calls a venue or archive. An optional saved capture check reads an explicitly supplied local path and is skipped otherwise.

The questions are deliberately operational:

1. Can the kernel identify the exact installed runtime and source/configuration inputs?
2. Which functional owner supplies each boundary?
3. Do configuration, UTC clocks, and exact Decimal arithmetic bind as expected?
4. Do replay and as-of consumers preserve cursor, correction, membership, and tamper contracts?
5. Does interrupted acquisition resume into publication and a pinned multi-input research handoff?
6. Does native capture recovery retain gaps, retries, and raw lineage?
7. Does supervision distinguish market, ingestion, heartbeat, child, queue, input, reconnect, and persistence failures?

Synthetic evidence validates software behavior only. It does not establish venue availability, source quality, strategy health, release readiness, or trading authority. Run all cells in order with the pinned interpreter described in the [operations guide](notebook-operations.md).
Use the [live version](infrastructure-diagnostics-live.ipynb) to initiate a new bounded public capture.
This offline version makes no provider request.


## 1. Setup and operating boundaries

**Question:** What state is this diagnostic allowed to create?

**Existing operation:** import the installed package, read maintained configuration/reference files, and run retained integration consumers in isolated child processes.

**Observed output:** the setup cell prints only the checkout, interpreter, and temporary workspace roots. Later cells print small tables; child command logs and raw fixture bytes stay in the temporary workspace.

**Interpretation:** every default result is reproducible without a saved input or network access. Acquisition itself remains in the existing CLI guide; this notebook exercises the already tested handoff using synthetic HTTP inside the retained consumer.

In [1]:
from __future__ import annotations

# standard library imports
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import timedelta
from decimal import Decimal
from importlib.metadata import distributions
from pathlib import Path
from tempfile import TemporaryDirectory

# third party library imports
import pandas as pd
from IPython.display import display

# project library imports
import market_data
from configuration import CaptureConfig, Configuration
from financial_units import Dimension, Price, Quantity, QuantityKind
from market_data.adapters.sources import BookAdapters
from market_data.contracts import CaptureRequest
from market_data.replay import SavedCapture
from reference_data.catalogs import CatalogSnapshot
from runtime import ImplementationIdentity, RuntimeFiles
from serialization import DomainCodec

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
assert (root / "configs").is_dir(), "Run from the repository root or examples/."

notebook_path = Path(
    os.environ.get(
        "ALPHA_RESEARCH_NOTEBOOK_PATH",
        root / "examples/infrastructure-diagnostics.ipynb",
    )
).resolve()
codec = DomainCodec()
workspace = TemporaryDirectory(prefix="infrastructure-diagnostics-")
local = Path(workspace.name)


class _NotebookTools:
    """Keep display and fixture copying local to this diagnostic notebook."""

    @staticmethod
    def show_table(headers: tuple[str, ...], rows: list[tuple[object, ...]]) -> None:
        """Render a compact table without truncating identities or reasons."""
        display(pd.DataFrame(rows, columns=headers))

    @staticmethod
    def stage(sources: tuple[Path, ...], destination: Path) -> None:
        """Copy the explicit non-secret inputs required by one existing consumer."""
        destination.mkdir()
        for source in sources:
            shutil.copyfile(source, destination / source.name)

    @staticmethod
    def tail(value: str, lines: int = 8) -> str:
        """Bound the displayed tail of an existing consumer's failure output."""
        return "\n".join(value.splitlines()[-lines:])


pd.set_option("display.max_colwidth", None)

print("Checkout:", root)
print("Kernel:", sys.executable)
print("Temporary workspace:", local)

Checkout: /private/tmp/alpha-research-live-offline-notebooks
Kernel: /Users/lxg/.cache/uv/builds-v0/.tmpGiPqgR/bin/python
Temporary workspace: /var/folders/30/8y6v8_j93_d8k3xzh930ghqw0000gn/T/infrastructure-diagnostics-wgqsc6_l


## 2. Runtime identity and functional ownership

The complete installed runtime inventory must match `src/` byte for byte. A stale
or editable installation fails before the consumers run; a matching package version
alone is insufficient. Record the pinned interpreter, platform, implementation,
lock, notebook source and actual tooling versions beside the execution attempt.

| Functional owner | Boundary to investigate |
| --- | --- |
| `configuration`, `reference_data` | Explicit jobs, canonical instrument IDs, authored intervals and units |
| `financial_units`, `serialization` | Exact dimensional arithmetic and wire contracts |
| `market_data.acquisition`, `datasets` | Verified checkpoints, quality admission and atomic publication |
| `market_data.collection`, `storage`, `books` | Durable receipt order, native state, gaps and recovery |
| `market_data.query`, `replay`, `research.exploration` | Cutoff selection, cursor contracts and pinned research assembly |
| `runtime`, `telemetry` | Independent supervision and bounded alert delivery |
| `applications` | Adapter selection and workflow composition |

Use the [architecture](../docs/architecture/platform-design.md) and
[test guide](../tests/README.md) for owning interfaces and focused checks.


In [2]:
assert platform.python_version() == (root / ".python-version").read_text().strip()
assert "site-packages" in Path(market_data.__file__).resolve().parts
source_root = root / "src"
installed_inputs = dict(RuntimeFiles.inputs())
source_files = {
    path.relative_to(source_root).as_posix(): path
    for package in RuntimeFiles.roots
    for path in (source_root / package).rglob("*")
    if path.is_file() and (path.suffix in {".py", ".json"} or path.name == "py.typed")
}
assert source_files.keys() == installed_inputs.keys(), "Installed runtime inventory differs"
for name, source in source_files.items():
    assert source.read_bytes() == installed_inputs[name].read_bytes(), (
        f"Stale installed file: {name}"
    )
notebook_path = Path(
    os.environ.get(
        "ALPHA_RESEARCH_NOTEBOOK_PATH", root / "examples/infrastructure-diagnostics.ipynb"
    )
)
notebook = json.loads(notebook_path.read_text())
notebook_source = [(cell["cell_type"], "".join(cell["source"])) for cell in notebook["cells"]]
identity = {
    "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip(),
    "working_tree": "changed"
    if subprocess.check_output(["git", "status", "--short"], cwd=root, text=True).strip()
    else "clean",
    "python": platform.python_version(),
    "platform": platform.platform(),
    "tooling": {
        item.metadata["Name"]: item.version
        for item in distributions()
        if item.metadata["Name"] in {"nbclient", "nbformat", "ipykernel", "pandas"}
    },
    "kernel": sys.executable,
    "package": market_data.__file__,
    "runtime_files": len(installed_inputs),
    "implementation_sha256": ImplementationIdentity.sha256(),
    "lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest(),
    "notebook_source_sha256": hashlib.sha256(
        json.dumps(notebook_source, ensure_ascii=False, separators=(",", ":")).encode()
    ).hexdigest(),
    "attempt": os.environ.get("ALPHA_RESEARCH_ATTEMPT_ID", "interactive: unsnapshotted"),
}
display(pd.DataFrame(identity.items(), columns=["Identity", "Value"]))

,Identity,Value
0,commit,00373626acc1459136163297cd4a76d2c0542787
1,working_tree,changed
2,python,3.14.7
3,platform,macOS-27.0-arm64-arm-64bit-Mach-O
4,tooling,"{'nbformat': '5.10.4', 'ipykernel': '7.2.0', 'nbclient': '0.10.4', 'pandas': '3.0.5'}"
5,kernel,/Users/lxg/.cache/uv/builds-v0/.tmpGiPqgR/bin/python
6,package,/private/tmp/alpha-research-live-offline-notebooks/.venv/lib/python3.14/site-packages/market_data/__init__.py
7,runtime_files,138
8,implementation_sha256,b67f60a16bd5f84fc3b0898389638dca4e16a7372418fa5fd3b0ad80c8d9c5f9
9,lock_sha256,e6cd3a169ff4b6850bcb5b09e38c44e1179b0dc1fb7c6a9bd95d2c4a058595e4


The map follows the architecture source map. It keeps application composition above reusable libraries and leaves strategy, risk, accounting, execution, deployment, and live activation outside this diagnostic because those capabilities are planned or require a separate consumer. A healthy source or collector process is not evidence of strategy health.

## 3. Configuration, reference, clock and numerical invariants

Resolve an authored catalog interval and inspect the configured job and capture limits.
The example computes `100.25 USDT / BTC × 2 BTC = 200.50 USDT` with dimensioned
Decimal values. This validates configuration and arithmetic, not historical venue
truth. No configuration is activated by reading it.


In [3]:
archive_config = Configuration.read(root / "configs/data-jobs/binance-archive.json")
fixture_config = Configuration.read(root / "configs/data-jobs/fixture-inspection.json")
catalog = codec.decode_json(
    (root / "configs/reference/binance-archive-spot.json").read_bytes(),
    CatalogSnapshot,
)
capture_config = CaptureConfig.read(root / "configs/live/bitfinex-btc-usd.json")
job = next(item for item in fixture_config.jobs if item.name == "btc-fixture")
instrument = catalog.resolve(catalog.instruments[0].instrument_id, job.start, job.end)
dimension = Dimension(
    instrument_id=instrument.instrument_id,
    kind=QuantityKind.BASE_UNIT,
)
notional = Price(
    value=Decimal("100.25"),
    currency=instrument.quote_currency,
    per=dimension,
).amount(Quantity(value=Decimal("2.00"), dimension=dimension))
checks = [
    (
        "fixture interval is UTC and one hour",
        job.start.utcoffset() == timedelta(0)
        and job.end.utcoffset() == timedelta(0)
        and job.end - job.start == timedelta(hours=1),
    ),
    ("catalog fully covers fixture interval", instrument.symbol == "BTCUSDT"),
    ("quote quantum is exact", instrument.quote_currency.minor_unit == Decimal("1E-8")),
    (
        "price times quantity remains exact",
        notional.value == Decimal("200.500000000000000000"),
    ),
    ("capture channel follows venue", capture_config.channel == "R0"),
    (
        "archive job remains explicitly configured",
        any(item.name == "spot-comparison" for item in archive_config.jobs),
    ),
]
assert all(result for _, result in checks)
_NotebookTools.show_table(
    ("Check", "Observed"),
    [
        ("fixture job", job.name),
        ("instrument", instrument.symbol),
        ("interval", f"{job.start.isoformat()} → {job.end.isoformat()}"),
        (
            "capture limits",
            f"{capture_config.duration_seconds}s / {capture_config.max_messages} messages",
        ),
        ("quote minor unit", instrument.quote_currency.minor_unit),
        ("exact notional", notional.value),
        *[(name, result) for name, result in checks],
    ],
)

,Check,Observed
0,fixture job,btc-fixture
1,instrument,BTCUSDT
2,interval,2024-06-01T00:00:00+00:00 → 2024-06-01T01:00:00+00:00
3,capture limits,20s / 10000 messages
4,quote minor unit,1E-8
5,exact notional,200.500000000000000000
6,fixture interval is UTC and one hour,True
7,catalog fully covers fixture interval,True
8,quote quantum is exact,True
9,price times quantity remains exact,True


## 4. Replay and as-of contracts

**Question:** Do the existing point-in-time and replay consumers keep source time, availability, phases, cursors, corrections, and integrity separate?

**Existing operation:** run the installed replay consumer against a tiny canonical partition, then run the installed as-of consumer against synthetic archive rows and authored membership/correction evidence. Both are copied outside the checkout and invoked with the isolated interpreter.

**Observed output:** the replay cell prints only source seconds, exact prices, batch/resume parity, and the bounded trace. The as-of cell prints only the correction cutoffs and tamper checks.

**Interpretation:** a cursor binds configuration identity and progress; an availability cutoff can legitimately return zero rows even when bytes exist. These consumers support causal research selection; they do not implement a strategy or an economic backtest.

In [4]:
replay_inputs = local / "replay-inputs"
_NotebookTools.stage(
    (
        root / "tests/integration/consumers/replay/replay_consumer.py",
        root / "configs/data-jobs/fixture-inspection.json",
        root / "configs/reference/binance-spot.json",
        root / "tests/fixtures/replay/canonical.json",
    ),
    replay_inputs,
)
replay_process = subprocess.run(
    [sys.executable, "-I", str(replay_inputs / "replay_consumer.py"), str(replay_inputs)],
    cwd=local,
    capture_output=True,
    text=True,
    check=False,
    timeout=30,
)
assert replay_process.returncode == 0, (
    replay_process.returncode,
    _NotebookTools.tail(replay_process.stdout),
    _NotebookTools.tail(replay_process.stderr),
)
replay_lines = [line for line in replay_process.stdout.splitlines() if line.strip()]
replay_observations = [
    line
    for line in replay_lines
    if line.startswith(
        ("Source seconds", "Batch sizes", "Canonical bytes", "Changed configuration")
    )
]
assert len(replay_observations) == 4
_NotebookTools.show_table(
    ("Replay observation", "Output"),
    [(str(index), line) for index, line in enumerate(replay_observations, 1)],
)

,Replay observation,Output
0,1,Changed configuration refused: incompatible or incomplete replay checkpoint
1,2,"Source seconds and exact prices: [(1, '100.00'), (3, 'disconnect'), (10, '101.00')]"
2,3,Batch sizes 1/8 and resumed trace agree; callback rows: 27
3,4,Canonical bytes: 17221 ; traced replay peak bytes: 122071


In [5]:
asof_inputs = local / "asof-inputs"
_NotebookTools.stage(
    (
        root / "tests/integration/consumers/query/asof_consumer.py",
        root / "configs/data-jobs/binance-archive.json",
        root / "configs/reference/binance-archive-spot.json",
        root / "tests/fixtures/eligibility/membership-corrections.json",
    ),
    asof_inputs,
)
asof_process = subprocess.run(
    [sys.executable, "-I", str(asof_inputs / "asof_consumer.py"), str(asof_inputs)],
    cwd=local,
    capture_output=True,
    text=True,
    check=False,
    timeout=30,
)
assert asof_process.returncode == 0, (
    asof_process.returncode,
    _NotebookTools.tail(asof_process.stdout),
    _NotebookTools.tail(asof_process.stderr),
)
asof_lines = [line for line in asof_process.stdout.splitlines() if line.strip()]
asof_observations = [
    line
    for line in asof_lines
    if line.startswith(
        ("Published synthetic snapshot", "Seconds / known eligible", "Exact revision")
    )
]
assert len(asof_observations) == 3
_NotebookTools.show_table(
    ("As-of observation", "Output"),
    [(str(index), line) for index, line in enumerate(asof_observations, 1)],
)

,As-of observation,Output
0,1,Published synthetic snapshot: 385eb27ff34afa1719fe942c8b6d48264ff28c5549e9ce11fb676fc08991819e
1,2,"Seconds / known eligible versions: [(0, 0), (5, 0), (9, 0), (10, 2), (43200, 1), (9, 0)]"
2,3,"Exact revision round trips, prior history, mutable-ID and tamper checks passed."


The correction and membership check is the handoff point for the research foundations notebook: bytes can be present while historical availability is still unknown, and a mutable snapshot alias or damaged canonical partition is refused. The result is synthetic contract evidence, not empirical eligibility evidence.

## 5. Acquisition, publication, interruption, and pinned research handoff

**Question:** Does one configured acquisition resume at its durable checkpoint and feed publication plus a pinned multi-input research selection?

**Existing operation:** use the retained ingestion_workflow.py consumer with its explicit required fixtures: the consumer, interrupt command, membership corrections, archive job, OKX capture config, and archive catalog. The consumer injects checksummed HTTP responses and exercises installed commands in child processes.

**Observed output:** the staging cell lists the copied inputs. The run cell reports the immutable snapshot and selection identities, research input count, quality scope, and nonzero interruption exits.

**Interpretation:** exit 130 preserves the first completed descriptor; incomplete publication exits 3; interruption before the rename exits 75; inspection of the interrupted candidate exits 4; the retry publishes the same identity. The resulting three-input selection reopens in a fresh process. This proves software handoff with synthetic data, not source coverage or historical acceptance.

In [6]:
handoff_inputs = local / "handoff-inputs"
_NotebookTools.stage(
    (
        root / "tests/integration/consumers/datasets/ingestion_workflow.py",
        root / "tests/support/subprocesses/interrupt_command.py",
        root / "tests/fixtures/eligibility/membership-corrections.json",
        root / "configs/data-jobs/binance-archive.json",
        root / "configs/live/okx-btc-usdt.json",
        root / "configs/reference/binance-archive-spot.json",
    ),
    handoff_inputs,
)
print("Staged handoff inputs:", ", ".join(sorted(path.name for path in handoff_inputs.iterdir())))

Staged handoff inputs: binance-archive-spot.json, binance-archive.json, ingestion_workflow.py, interrupt_command.py, membership-corrections.json, okx-btc-usdt.json


In [7]:
handoff_directory = local / "handoff"
handoff_process = subprocess.run(
    [
        sys.executable,
        "-I",
        str(handoff_inputs / "ingestion_workflow.py"),
        str(handoff_inputs),
        str(handoff_directory),
    ],
    cwd=local,
    capture_output=True,
    text=True,
    check=False,
    timeout=75,
)
assert handoff_process.returncode == 0, (
    handoff_process.returncode,
    _NotebookTools.tail(handoff_process.stdout),
    _NotebookTools.tail(handoff_process.stderr),
)
handoff_result = json.loads((handoff_directory / "result.json").read_text())
handoff_commands = [
    json.loads(line) for line in (handoff_directory / "commands.jsonl").read_text().splitlines()
]
nonzero_exits = [item["exit_code"] for item in handoff_commands if item["exit_code"]]
assert nonzero_exits == [130, 3, 75, 4], nonzero_exits
assert handoff_result["research_inputs"] == 3
assert handoff_result["repeat_fetches"] == 0
_NotebookTools.show_table(
    ("Handoff field", "Observed"),
    [
        ("configured job", handoff_result["configured_job"]),
        ("snapshot identity", handoff_result["snapshot_id"]),
        ("selection identity", handoff_result["selection_identity"]),
        ("partitions", handoff_result["partitions"]),
        ("research inputs", handoff_result["research_inputs"]),
        ("scope", handoff_result["scope"]),
        (
            "history / cutoff rows",
            f"{handoff_result['history_rows']} / {handoff_result['cutoff_rows']}",
        ),
        ("resume first descriptor", handoff_result["resume_preserved_first_descriptor"]),
        ("repeat source fetches", handoff_result["repeat_fetches"]),
        ("nonzero child exits", nonzero_exits),
    ],
)

,Handoff field,Observed
0,configured job,spot-comparison
1,snapshot identity,998da6dbd609e3681de0f27772ddd1e4ac4811f24db9b9a3bd0e142e3ecbb46d
2,selection identity,984ba3f05781085b38362179a5c04fa749cf2f5a934a4b419ff73e86ffc0935d
3,partitions,2
4,research inputs,3
5,scope,exploratory_degraded
6,history / cutoff rows,6 / 0
7,resume first descriptor,True
8,repeat source fetches,0
9,nonzero child exits,"[130, 3, 75, 4]"


This is the retained pinned research handoff used by the QR path. The selected trade inputs and synthetic OKX book are named in selection.json, reopened through the installed EDA workflow, and kept separate by type. Research interpretation should retain the printed identities and scope; it should not treat a successful handoff as a proof of source quality.

## 6. Native capture recovery

**Question:** When a native session disconnects or recovery fails, do durable records retain the unknown gap and later receipt order?

**Existing operation:** run recovery_consumer.py with its copied Coinbase schema, snapshot, and capture configuration. The consumer injects late snapshots, bridge gaps, rate limits, exact duplicates, conflicting repeats, stale snapshots, and bounded backoff through the existing transport boundary.

**Observed output:** the next cell reports duplicate/stale lineage and one compact row per recovery mode.

**Interpretation:** a later snapshot does not repair an earlier unknown gap or acquire its earlier receipt time. Replay parity means the saved journal reproduces the collector's observed state; it does not mean the source was complete or that reconnects are exactly once.

In [8]:
recovery_inputs = local / "recovery-inputs"
_NotebookTools.stage(
    (
        root / "tests/integration/consumers/capture/recovery_consumer.py",
        root / "tests/fixtures/market_data/coinbase/coinbase-schema.json",
        root / "tests/fixtures/market_data/coinbase/coinbase-snapshot.json",
        root / "configs/live/coinbase-btc-usd.json",
    ),
    recovery_inputs,
)
recovery_directory = local / "session-recovery"
recovery_process = subprocess.run(
    [
        sys.executable,
        "-I",
        str(recovery_inputs / "recovery_consumer.py"),
        str(recovery_inputs),
        str(recovery_directory),
    ],
    cwd=local,
    capture_output=True,
    text=True,
    check=False,
    timeout=45,
)
assert recovery_process.returncode == 0, (
    recovery_process.returncode,
    _NotebookTools.tail(recovery_process.stdout),
    _NotebookTools.tail(recovery_process.stderr),
)
recovery_result = json.loads((recovery_directory / "result.json").read_text())
assert Path(recovery_result["installed_package"]).resolve() == Path(market_data.__file__).resolve()
duplicate = recovery_result["duplicate_and_stale"]
_NotebookTools.show_table(
    ("Lineage check", "Observed"),
    [
        ("duplicate receipt clocks", duplicate["duplicate_receipt_ns"]),
        ("economic matches", duplicate["economic_matches"]),
        ("conflicting repeat", duplicate["conflicting_duplicate"]),
        ("stale snapshots", duplicate["stale_snapshots"]),
        ("new epoch synchronized", duplicate["new_epoch_synchronized"]),
    ],
)
_NotebookTools.show_table(
    (
        "Scenario",
        "connections",
        "snapshot requests",
        "backoff",
        "raw snapshots",
        "retry exhausted",
        "synchronized",
        "replay parity",
    ),
    [
        (
            row["scenario"],
            row["connections"],
            row["snapshot_requests"],
            row["backoff_seconds"],
            row["retained_snapshots"],
            row["retry_exhausted"],
            row["synchronized"],
            row["replay_parity"],
        )
        for row in recovery_result["recovery"]
    ],
)

,Lineage check,Observed
0,duplicate receipt clocks,"[5, 6]"
1,economic matches,1
2,conflicting repeat,invalid_frame
3,stale snapshots,2
4,new epoch synchronized,False


,Scenario,connections,snapshot requests,backoff,raw snapshots,retry exhausted,synchronized,replay parity
0,late,3,3,"[1.0, 2.0]",3,True,False,True
1,bridge_gap,3,3,"[1.0, 2.0]",3,True,False,True
2,rate_limit,3,3,"[1.0, 2.0]",0,True,False,True


## 7. Supervision health

**Question:** Can the watchdog distinguish progress clocks and child failures while retaining an interrupted journal prefix?

**Existing operation:** run health_consumer.py with a copied Bitfinex config. It supplies synthetic R0 frames through the existing collector, receives alerts on a temporary local Unix datagram socket, and runs each bounded failure mode in a child-controlled output directory.

**Observed output:** the next cell shows stop reasons, child termination, alert delivery, and committed frame counts.

**Interpretation:** market_stalled means heartbeats continue while source freshness stops; persistence_stalled or ingestion_stalled means downstream progress stopped; heartbeat_stalled means the event loop stopped; collector_lost means the child died; persistence_failed, invalid_limit, reconnects_limit, and queue_limit identify their configured controls. Local socket delivery proves only that the notifier emitted a bounded message. Source health is separate from strategy health, risk, accounting, and execution.

In [9]:
health_inputs = local / "health-inputs"
_NotebookTools.stage(
    (
        root / "tests/integration/consumers/capture/health_consumer.py",
        root / "configs/live/bitfinex-btc-usd.json",
    ),
    health_inputs,
)
health_directory = local / "health-consumer"
health_process = subprocess.run(
    [
        sys.executable,
        "-I",
        str(health_inputs / "health_consumer.py"),
        str(health_inputs / "bitfinex-btc-usd.json"),
        str(health_directory),
    ],
    cwd=local,
    capture_output=True,
    text=True,
    check=False,
    timeout=60,
)
assert health_process.returncode == 0, (
    health_process.returncode,
    _NotebookTools.tail(health_process.stdout),
    _NotebookTools.tail(health_process.stderr),
)
health_evidence = json.loads(health_process.stdout)
assert Path(health_evidence["package"]).resolve() == Path(market_data.__file__).resolve()
health_rows = health_evidence["observations"]
assert all(row["collector_stopped"] for row in health_rows)
assert any(row["stop_reason"] == "heartbeat_stalled" for row in health_rows)
assert any(row["stop_reason"] == "collector_lost" for row in health_rows)
assert any(
    row["stop_reason"] in {"ingestion_stalled", "persistence_stalled"} for row in health_rows
)
_NotebookTools.show_table(
    ("Scenario", "Stop reason", "Child stopped", "Alert delivered", "Committed frames"),
    [
        (
            row["scenario"],
            row["stop_reason"],
            row["collector_stopped"],
            row["alert_delivered"],
            row["committed_frames"],
        )
        for row in health_rows
    ],
)

,Scenario,Stop reason,Child stopped,Alert delivered,Committed frames
0,healthy,completed,True,False,3
1,market,market_stalled,True,True,52
2,ingestion,persistence_stalled,True,True,3
3,blocked,heartbeat_stalled,True,True,3
4,death,collector_lost,True,True,3
5,disk,persistence_failed,True,True,3
6,invalid,invalid_limit,True,True,4
7,reconnect,reconnects_limit,True,True,4
8,queue,queue_limit,True,True,2
9,alert_failure,market_stalled,True,False,50


## 8. Optional saved native diagnostic

**Question:** Is there a specific completed or interrupted native recording that needs a local replay check?

**Existing operation:** if ALPHA_BITFINEX_CAPTURE_DIRECTORY names an explicit saved recording, SavedCapture.read() verifies its request and journals through BookAdapters.create. No source request is made.

**Observed output:** the default label is SKIPPED. With an explicit path, the cell prints only the recording summary.

**Interpretation:** use the existing capture and replay CLI guide to acquire or pin inputs. A saved capture can show bounded reconstructed state and gaps; it does not establish continuous source coverage.

Completed mode is required by default. Change `completed` to `False` only for an
explicit interrupted-prefix investigation; a missing manifest is never an automatic
mode switch. Invalid supplied paths and damaged bytes fail visibly.


In [10]:
capture_path_text = os.environ.get("ALPHA_BITFINEX_CAPTURE_DIRECTORY", "").strip()
if not capture_path_text:
    print(
        "SKIPPED: ALPHA_BITFINEX_CAPTURE_DIRECTORY is unset; no saved native input was requested."
    )
else:
    capture_path = Path(capture_path_text).expanduser().resolve()
    assert capture_path.is_dir(), capture_path
    request = codec.decode_json(
        (capture_path / "request.json").read_bytes(),
        CaptureRequest,
    )
    assert request.config.venue == "bitfinex", "Expected an explicitly selected Bitfinex capture"
    completed = True  # Set False only when deliberately diagnosing an interrupted prefix.
    replay = SavedCapture.read(
        capture_path,
        book_factory=BookAdapters.create,
        completed=completed,
    )
    summary = replay.summary()
    _NotebookTools.show_table(
        ("Saved capture field", "Observed"),
        [
            ("venue / symbol", f"{summary['venue']} / {summary['symbol']}"),
            ("recording identity", request.recording_id),
            ("completed manifest", completed),
            ("messages / frames", f"{summary['messages']} / {summary['frames']}"),
            ("epochs / gaps", f"{summary['epochs']} / {len(summary['gaps'])}"),
            ("synchronized", summary["synchronized"]),
            ("quality", summary["quality"]),
        ],
    )

SKIPPED: ALPHA_BITFINEX_CAPTURE_DIRECTORY is unset; no saved native input was requested.


## 9. Historical workload record and release boundary

Read the [retained measurement](evidence/query-workload-measurements.json) with its
original revision, runtime, driver, lock and snapshot identities. These fingerprints
identify a historical workload; the current runtime has a different package layout.
Do not treat a comparison of these hash contracts as fresh performance evidence.
No timing driver or large raw dataset is loaded. Follow the
[reproduction procedure](notebook-operations.md#query-workload-evidence) for a demanded
new workload. Existing [release restrictions](../docs/architecture/README.md#dependency-and-release-boundaries)
remain separate from development diagnostics.


In [11]:
measurement_path = root / "examples/evidence/query-workload-measurements.json"
measurement = json.loads(measurement_path.read_text())
for field in (
    "schema",
    "measured_at",
    "runtime_commit",
    "runtime_sha256",
    "driver_sha256",
    "lock_sha256",
    "snapshot_id",
    "budgets",
):
    assert field in measurement
_NotebookTools.show_table(
    ("Historical field", "Recorded value"),
    [
        ("schema", measurement["schema"]),
        ("measured at", measurement["measured_at"]),
        ("runtime commit", measurement["runtime_commit"]),
        ("runtime SHA-256", measurement["runtime_sha256"]),
        ("driver SHA-256", measurement["driver_sha256"]),
        ("lock SHA-256", measurement["lock_sha256"]),
        ("snapshot identity", measurement["snapshot_id"]),
        ("budgets", measurement["budgets"]),
    ],
)
print("No fresh benchmark was run; the retained record remains historical evidence.")

,Historical field,Recorded value
0,schema,mr18-workload-evaluation-v1
1,measured at,2026-09-17T12:15:18.739620+00:00
2,runtime commit,cae1b06
3,runtime SHA-256,7c03512d77fa1152dc7feb945d708c3a2e794599eb00e09546b204c57c29f9e7
4,driver SHA-256,f1409f02b0315683cc9d5ef767e6d49549b67403d578fbbf3a6007c0e028b48a
5,lock SHA-256,e6cd3a169ff4b6850bcb5b09e38c44e1179b0dc1fb7c6a9bd95d2c4a058595e4
6,snapshot identity,c0f8f3953dee8a15a79d1e9a597e37220c094b3d3c555c8f06d0ec366d3ed278
7,budgets,"{'load_seconds': 300, 'query_p95_seconds': 30, 'peak_rss_bytes': 1073741824}"


No fresh benchmark was run; the retained record remains historical evidence.


## 10. Engineer reading of the result

The successful cells establish a bounded offline software path:

- installed bytes and source inventory are identical for the checked runtime inputs;
- reference/configuration and exact numerical contracts identify UTC intervals and dimensioned values;
- replay and as-of consumers preserve point-in-time and cursor semantics;
- acquisition resumes through atomic publication into a pinned three-input research handoff;
- native recovery retains unknown loss and later receipt order;
- supervision stops distinct failure modes and records whether local alert delivery occurred.

The release and research limits remain explicit. Synthetic inputs do not become empirical evidence. Capture health does not become strategy health. A complete publication does not make historical availability known. Query/replay is not a backtest, and a saved book is not an executable fill model. The retained workload record is not requalified here.

For an unexpected failure, stop at its first failing cell. Inspect the child error
and owning component rather than changing an expected result. Use the
[test guide](../tests/README.md) for the affected consumer and the source guides for
actual acquisition. The optional native path is explicitly skipped when unset.
Temporary synthetic stores are removed below. Saved captures are read only; retain
real diagnostic evidence separately before replacing a conclusion. The notebook runner
preserves completed and failed local attempts, which are not a backed-up incident archive.


In [12]:
findings = [
    ("Runtime identity", "PASS", "installed inventory matches src/"),
    ("Config and exact values", "PASS", "UTC, Decimal, and venue limits are explicit"),
    ("Replay and as-of", "PASS", "cursor, cutoff, correction, and tamper checks passed"),
    ("Pinned handoff", "PASS", "resume/publication/selection consumer passed"),
    ("Native recovery", "PASS", "gaps and raw lineage remain visible"),
    ("Supervision", "PASS", "bounded health stop reasons were observed"),
    ("Empirical/release qualification", "OUT OF SCOPE", "requires separate evidence and authority"),
]
_NotebookTools.show_table(("Diagnostic area", "Status", "Meaning"), findings)
print("Synthetic temporary stores removed; supplied saved captures were read only.")
workspace.cleanup()

,Diagnostic area,Status,Meaning
0,Runtime identity,PASS,installed inventory matches src/
1,Config and exact values,PASS,"UTC, Decimal, and venue limits are explicit"
2,Replay and as-of,PASS,"cursor, cutoff, correction, and tamper checks passed"
3,Pinned handoff,PASS,resume/publication/selection consumer passed
4,Native recovery,PASS,gaps and raw lineage remain visible
5,Supervision,PASS,bounded health stop reasons were observed
6,Empirical/release qualification,OUT OF SCOPE,requires separate evidence and authority


Synthetic temporary stores removed; supplied saved captures were read only.
